In [25]:
import random

class NakliLLM:
    def __init__(self):
        print("LLM Created")
    
    def predict(self,prompt):

        respose_list=[
            'Delhi is the capital of India',
            'IPL is a cricket league',
            'AI stands for artificila Intelligence'
        ]

        return{'response':random.choice(respose_list)}

In [26]:
class NakliPromttemplate:
    def __init__(self,template,input_variables):
        self.template= template
        self.input_variables= input_variables

    def format(self,input_dict):
        return self.template.format(**input_dict)
            


In [27]:
template=NakliPromttemplate(
    template="write a {length} poem about {topic}",
    input_variables=['length','topic'])

prompt = template.format({'length':'short','topic':'india'})

In [28]:
llm=NakliLLM()

LLM Created


In [29]:
llm.predict(prompt)

{'response': 'Delhi is the capital of India'}

In [30]:
class NakliLmChain:
    def __init__(self,llm,prompt):
        self.llm=llm
        self.prompt =prompt

    def run(self,input_dict):
        final_prompt=self.prompt.format(input_dict)
        result=self.llm.predict(final_prompt)

        return result['response']

In [31]:
template=NakliPromttemplate(
    template="write a {length} poem about {topic}",
    input_variables=['length','topic'])
llm=NakliLLM()

LLM Created


In [32]:
chain=NakliLmChain(llm,template)

In [33]:
chain.run({'length':'short','topic':'india'})

'IPL is a cricket league'

In [34]:
#This is not flexible, we cannot make two calls, it is very diificult to do

#The way to interact with classes are different

## Solution

In [54]:
from abc import ABC, abstractmethod

#Abstract based method

In [55]:
class Runnable(ABC):

    @abstractmethod
    def invoke(input_data):
        pass

In [69]:
class NakliLLM(Runnable):
    def __init__(self):
        print("LLM Created")
    
    def invoke(self,prompt):

        respose_list=[
            'Delhi is the capital of India',
            'IPL is a cricket league',
            'AI stands for artificila Intelligence'
        ]

        return{'response':random.choice(respose_list)}
    
    def predict(self,prompt):

        respose_list=[
            'Delhi is the capital of India',
            'IPL is a cricket league',
            'AI stands for artificila Intelligence'
        ]

        return{'response':random.choice(respose_list)}
    
class NakliPromttemplate(Runnable):
    def __init__(self,template,input_variables):
        self.template= template
        self.input_variables= input_variables

    def invoke(self,input_dict):
        return self.template.format(**input_dict)    

    def format(self,input_dict):
        return self.template.format(**input_dict)
    
class NakliStrOutoutParser(Runnable):
    def __init__(self,):
        pass

    def invoke(self,input_data):
        return(input_data['response'])
            
  

In [70]:
class RunnableConnector(Runnable):
    def __init__(self,runnable_list):
        self.runnable_list=runnable_list
   

    def invoke(self,input_data):

        for runnable in self.runnable_list:
            input_data=runnable.invoke(input_data)  

        return input_data

In [71]:
template=NakliPromttemplate(
    template="write a {length} poem about {topic}",
    input_variables=['length','topic'])
llm=NakliLLM()
parser=NakliStrOutoutParser()
chain=RunnableConnector([template,llm,parser])


LLM Created


In [ ]:
# chain.invoke({'length':'short','topic':'india'})

'Delhi is the capital of India'

In [ ]:
#Now we will connect multiple chains to make a big chain

template1=NakliPromttemplate(
    template='Wrte a joke about {topic}',
    input_variables=['topic']
)

template2=NakliPromttemplate(
    template='Explain the following Joke {response}',
    input_variables=['response']
)

llm = NakliLLM()

LLM Created


In [85]:
parser=NakliStrOutoutParser()

chain1=RunnableConnector([template1,llm])
chain2= RunnableConnector([template2,llm,parser])

final_chain=RunnableConnector([chain1,chain2])


In [ ]:
chain1.invoke({'topic':'AI'})
chain2.invoke({'response':'This is a joke'})

{'response': 'IPL is a cricket league'}

In [86]:
final_chain.invoke({'topic':'AI'})

'IPL is a cricket league'

'AI stands for artificila Intelligence'